In [1]:
# ============ GEMMA3 TRANSLATION + CONCAT (FR DEV) ============
import os, re, json, requests
from datetime import datetime
from tqdm import tqdm
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, Dataset

def find_cache_dir():
    for c in ['baseline/cache', 'cache', '../baseline/cache']:
        if os.path.exists(c): return c
CACHE_DIR = find_cache_dir()
RESULTS_DIR = os.path.join(os.path.dirname(CACHE_DIR), 'results')

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
batch_size = 16
print(f"Device: {device}")

collection_embeddings = torch.load(os.path.join(CACHE_DIR, 'collection_embeddings.pt'))
collection_keys = np.load(os.path.join(CACHE_DIR, 'collection_keys.npy'))
e5_name = 'intfloat/multilingual-e5-large'
tok = AutoTokenizer.from_pretrained(e5_name)
model = AutoModel.from_pretrained(e5_name).to(device).eval()
print("E5 loaded.")

OLLAMA = "http://localhost:11434/api/generate"
MT_MODEL = "gemma3:1b"

def avg_pool(h, m):
    h = h.masked_fill(~m[..., None].bool(), 0.0)
    return h.sum(dim=1) / m.sum(dim=1)[..., None]

def clean_tweet(t):
    t = re.sub(r'http\S+|www\.\S+', '', t)
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#(\w+)', r'\1', t)
    t = re.sub(r'^\d+/n\s*Fact\s*\d+\s*:\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^\d+/\d+\s*', '', t)
    t = re.sub(r'[^\w\s\.\,\:\;\!\?\-\(\)\']+', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def gemma_translate(text, src_lang_name, retries=2):
    prompt = (
        f"Translate the following {src_lang_name} sentence into English. "
        f"Output ONLY the English translation, no preamble or explanation.\n\n"
        f"{src_lang_name}: {text}\n\nEnglish:"
    )
    payload = {
        "model": MT_MODEL,
        "prompt": prompt,
        "stream": False,
        "keep_alive": "60m",
        "options": {"temperature": 0.0, "num_predict": 200}
    }
    for attempt in range(retries + 1):
        try:
            r = requests.post(OLLAMA, json=payload, timeout=60)
            r.raise_for_status()
            return r.json()["response"].strip().split("\n")[0]
        except Exception:
            if attempt == retries: return text
            continue

def embed(texts, max_len=160):
    ds = Dataset.from_dict({"text": [f'query: {t}' for t in texts]})
    ds = ds.map(lambda e: tok(e["text"], max_length=max_len, truncation=True, padding="max_length", return_tensors="pt"),
                batched=True, remove_columns=ds.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
    embs = []
    with torch.inference_mode():
        for b in tqdm(DataLoader(ds, batch_size=batch_size, num_workers=0), desc="Embed"):
            b = {k: v.to(device) for k, v in b.items()}
            embs.append(avg_pool(model(**b).last_hidden_state, b['attention_mask']).cpu())
    return F.normalize(torch.vstack(embs), p=2, dim=1)

def mrr5(cs, lab):
    top5 = np.argsort(-cs, axis=1)[:, :5]
    keys = collection_keys[top5]
    m = (keys == lab[:, None])
    pos = np.argmax(m, axis=1); pos[~m.any(axis=1)] = -1
    rr = np.zeros_like(pos, dtype=float); rr[pos >= 0] = 1.0 / (pos[pos >= 0] + 1)
    return float(rr.mean())

# === FR Dev ===
data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "fr")
dev_raw = data["dev"]
dev_labels = np.array(dev_raw["pubkey"])
dev_clean = [clean_tweet(t) for t in dev_raw["text"]]

print(f"\nGemma3 translating {len(dev_clean)} FR dev tweets...")
dev_gemma_en = []
for t in tqdm(dev_clean, desc="Gemma3 FR->EN"):
    dev_gemma_en.append(gemma_translate(t, "French"))

# Save
with open(os.path.join(CACHE_DIR, 'dev_en_fr_gemma3.json'), 'w', encoding='utf-8') as f:
    json.dump(dev_gemma_en, f, ensure_ascii=False)

for i in range(3):
    print(f"\nFR:    {dev_clean[i][:130]}")
    print(f"GEMMA: {dev_gemma_en[i][:130]}")

# Variant 1: Gemma3 alone (replace, like Cam)
print("\n--- Variant 1: Gemma3 only ---")
emb1 = embed(dev_gemma_en)
score1 = mrr5((emb1 @ collection_embeddings.T).numpy(), dev_labels)
print(f"MRR@5 (Gemma3 only): {score1:.4f}")

# Variant 2: clean + Gemma3 concat
print("\n--- Variant 2: clean + Gemma3 concat ---")
dev_concat = [f"{c} {e}" for c, e in zip(dev_clean, dev_gemma_en)]
emb2 = embed(dev_concat)
score2 = mrr5((emb2 @ collection_embeddings.T).numpy(), dev_labels)
print(f"MRR@5 (clean + Gemma3 concat): {score2:.4f}")

print("\n=== FR DEV COMPARISON ===")
print(f"Baseline:           0.4584")
print(f"Cleaning:           0.4620")
print(f"Concat (opus-mt):   0.4864  ← my best")
print(f"Gemma3 only:        {score1:.4f}  (Cam's strategy)")
print(f"Concat (Gemma3):    {score2:.4f}  ← potentially new best")

torch.save(emb2, os.path.join(CACHE_DIR, 'dev_embeddings_fr_gemma_concat.pt'))
log = {
    "timestamp": datetime.now().isoformat(),
    "experiment": "exp11_gemma3_fr_dev",
    "language": "fr",
    "gemma3_only_mrr5_dev": score1,
    "gemma3_concat_mrr5_dev": score2,
    "delta_concat_vs_opusmt": score2 - 0.4864,
}
with open(os.path.join(RESULTS_DIR, 'results.jsonl'), 'a') as f:
    f.write(json.dumps(log) + '\n')
print("\nLogged.")

/Users/ekinkeskin/air2026ss-checkthat-task1-checkthat-task1-methodology-3-group-1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7945.76it/s]


E5 loaded.

Gemma3 translating 702 FR dev tweets...


Gemma3 FR->EN: 100%|██████████| 702/702 [11:47<00:00,  1.01s/it]



FR:    Psychose sévère provoquée par une inflammation cérébrale des récepteurs anti-N-méthyl D-aspartate suite à fpizerisation : minotte 
GEMMA: Severe psychosis caused by cerebral inflammation of N-methyl D-aspartate receptor following Pfizer infection: minotté of 20 years

FR:    L Indomethacin (anti-inflammatoire non stéroïdien) employé conjointement au plan de traitement habituel chez les patients hospital
GEMMA: Indomethacin (non-steroidal anti-inflammatory drug) used in combination with the usual treatment regimen in hospitalized patients 

FR:    splendides photos : Étude sur la pollution des masques, analysée par des experts -à présent en ligne: La pollution des masques par
GEMMA: Splendid photos: Study on the pollution of masks, analyzed by experts – now available: The pollution of masks by bacteria and fung

--- Variant 1: Gemma3 only ---


Embed: 100%|██████████| 44/44 [01:02<00:00,  1.42s/it]


MRR@5 (Gemma3 only): 0.4969

--- Variant 2: clean + Gemma3 concat ---


Embed: 100%|██████████| 44/44 [01:08<00:00,  1.56s/it]


MRR@5 (clean + Gemma3 concat): 0.4877

=== FR DEV COMPARISON ===
Baseline:           0.4584
Cleaning:           0.4620
Concat (opus-mt):   0.4864  ← my best
Gemma3 only:        0.4969  (Cam's strategy)
Concat (Gemma3):    0.4877  ← potentially new best

Logged.
